# MIND — an AI that learns from its own mistakes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anusidhu9/ai-that-learns-from-its-mistakes/blob/main/MIND_Colab.ipynb)

A reinforcement-learning agent explores a grid world. It is **rewarded** for reaching the goal and **penalised** for mistakes: hitting walls, stepping on hazards, revisiting cells, and running out of time. This notebook trains three agents on the same maze and checks whether their mistakes actually go down:

| Agent | Idea |
|---|---|
| Random | Baseline that never learns |
| Tabular Q-learning | A table of action values, updated by $Q(s,a) \leftarrow Q(s,a) + \alpha\,[\,r + \gamma \max_{a'} Q(s',a') - Q(s,a)\,]$ |
| DQN | A neural network approximates $Q$, trained on replayed past experience with a frozen target network |

Runs on the **free Colab CPU**; no GPU needed. Total runtime is about 2–3 minutes.

> Sections 1–10 train and inspect agents, 11 shows the multi-seed evidence, 12 adds the lighter side (nicknames, achievements, Hall of Shame), 13 draws the saved results, and 14 launches the interactive **MIND dashboard** website.

## 1. Setup
Clones the repository (or updates it if already cloned) and makes `src` importable.

> **Updating an open session:** Python keeps already-imported code in memory, so after new code is pushed, use **Runtime → Restart session** and then **Run all**. Running `git pull` alone is not enough, and a dashboard launched earlier keeps showing the old version until it is relaunched.

In [ ]:
import os, sys

REPO_URL = "https://github.com/anusidhu9/ai-that-learns-from-its-mistakes.git"
REPO_DIR = "ai-that-learns-from-its-mistakes"

if not os.path.exists("src/envs.py"):          # not already inside the repo
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    %cd {REPO_DIR}
!git pull -q
!pip install -q pytest gradio

sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())
print("Code version:", os.popen('git log -1 --format="%h (%cd)" --date=short').read().strip(),
      "(compare with the latest commit on GitHub)")

## 2. Run the test suite
The quick subset takes about 15 s. Every test should pass; the exact count grows as stages are added.

In [ ]:
!python -m pytest -q -m "not slow"

## 3. The environments

Four presets share one observation format, so every algorithm sees exactly the same information.
`#` wall · `.` free · `H` hazard · `G` goal · `A` agent. Every layout is guaranteed to have a hazard-free path to the goal.

In [ ]:
import time
import numpy as np
import pandas as pd
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import display   # charts appear inline

from src.envs import make_env, PRESETS
from src.agents import make_agent, EpsilonSchedule
from src.training import train_agent

for name in PRESETS:
    env = make_env(name)
    env.reset(seed=0)
    print(f"{name}: {env.config.height}x{env.config.width}, shortest safe path = "
          f"{env.optimal_length} steps, observation size = {env.obs_size}")
    print(env.render_text(), end="\n\n")

## 4. Shared settings
Charts come from `src/visualization.py`, the same code the dashboard uses, so the notebook and the website always draw them the same way.

In [ ]:
from src.visualization import AGENT_COLORS, draw_grid, learning_curves_figure, mistake_heatmap_figure

PRESET = "maze"        # try "maze_hazards", "open", "open_hazards"
EPISODES = 400
SEED = 0
WINDOW = 50            # episodes compared at the start vs the end of training
AGENTS = [("Random", "random"), ("Tabular Q", "tabular_q"), ("DQN", "dqn")]

## 5. Train the three agents
Tabular Q-learning takes a few seconds; DQN takes 1–2 minutes on the free CPU.

In [ ]:
results = {}
schedule = dict(decay_episodes=300)   # epsilon: 1.0 -> 0.05 over 300 episodes

for label, agent_name in AGENTS:
    env = make_env(PRESET)
    kwargs = {} if agent_name == "random" else {"epsilon": EpsilonSchedule(**schedule)}
    agent = make_agent(agent_name, env.obs_size, env.n_actions, seed=SEED, **kwargs)
    start = time.time()
    history = train_agent(env, agent, EPISODES, seed=SEED)
    results[label] = {"env": env, "agent": agent, "history": history, "name": agent_name}
    print(f"{label:<10} trained {EPISODES} episodes in {time.time() - start:5.1f} s")

## 6. Learning curves

Each line is a 25-episode rolling mean. **Success rate is the primary metric.** A falling mistake count alone can be misleading; for example, an agent that ends episodes early on a hazard also makes fewer mistakes.

In [ ]:
display(learning_curves_figure({res["name"]: res["history"] for res in results.values()}))

## 7. Early vs late training
Mean over the first and last 50 training episodes (exploration is still partly on during training).

In [ ]:
rows = []
for label, res in results.items():
    h = res["history"]
    for phase, part in [(f"first {WINDOW}", h[:WINDOW]), (f"last {WINDOW}", h[-WINDOW:])]:
        rows.append({
            "agent": label, "phase": phase,
            "success rate": np.mean([e["success"] for e in part]),
            "steps": np.mean([e["steps"] for e in part]),
            "total reward": np.mean([e["total_reward"] for e in part]),
            "collisions": np.mean([e["collisions"] for e in part]),
            "hazards": np.mean([e["hazards"] for e in part]),
            "revisits": np.mean([e["revisits"] for e in part]),
            "timeouts": np.mean([e["timeout"] for e in part]),
        })
pd.DataFrame(rows).set_index(["agent", "phase"]).round(2)

## 8. Greedy evaluation (exploration off)

The real test: with ε = 0 the agent must rely only on what it learned. `optimal` is the shortest hazard-free path found by breadth-first search.

In [ ]:
def greedy_path(env, agent):
    obs, _ = env.reset()
    path, done = [env.pos], False
    while not done:
        obs, _, term, trunc, info = env.step(agent.act(obs, greedy=True))
        path.append(env.pos)
        done = term or trunc
    return path, info["episode"]

fig, axes = plt.subplots(1, len(results), figsize=(4.2 * len(results), 4.8), layout="constrained")
table = []
for ax, (label, res) in zip(axes, results.items()):
    path, ep = greedy_path(res["env"], res["agent"])
    table.append({"agent": label, "success": ep["success"], "steps": ep["steps"],
                  "optimal": ep["optimal_length"], "mistakes": ep["mistakes"]})
    outcome = "reached the goal" if ep["success"] else "failed"
    # two-line titles so neighbouring panel titles never run into each other
    draw_grid(ax, res["env"].grid, start=res["env"].start, pos=path[-1], path=path,
              path_color=AGENT_COLORS[res["name"]], title=f"{label}\n{outcome} in {ep['steps']} steps")
plt.show()
pd.DataFrame(table).set_index("agent")

## 9. Where does the agent make mistakes?

Mistakes per cell (collisions, revisits, hazards), first 50 vs last 50 training episodes of tabular Q-learning. Both panels share one colour scale, so a fading hotspot means the agent stopped repeating that mistake.

In [ ]:
res = results["Tabular Q"]
h, env = res["history"], res["env"]
mistakes = [{**m, "episode": e["episode"]} for e in h for m in e["mistake_log"]]
display(mistake_heatmap_figure(env.grid, mistakes, range(0, WINDOW),
                               range(EPISODES - WINDOW, EPISODES), start=env.start))

## 10. What this single run does and doesn't show

* This is **one seed on one fixed layout**. It shows that learning happens, but it is not evidence about reliability or generalisation.
* Mistakes during training are measured with exploration still partly on; section 8 is the fair, exploration-free check.
* Section 11 has the multi-seed evaluation that the project's claims rest on.

## 11. Multi-seed evaluation (Stage 5)

Each experiment trains every agent with several seeds, then evaluates it **greedily (exploration off)**. Numbers are the mean across seeds ± a 95% confidence interval; with 5 seeds the interval is wide, so treat small differences as unresolved. The saved results in `results/` were produced with:

```
python -m src.evaluation --experiment <name>
```

| Experiment | Question |
|---|---|
| `fixed_maze` | Can each agent solve one fixed maze without mistakes? |
| `hazard_maze` | Same, with 4 hazards that cost −0.5 but don't end the episode |
| `terminal_hazards` | Same, but a hazard ends the episode |
| `open_hazards` | Open 8×8 grid with 5 hazards |
| `generalization` | Train on 50 mazes (local 5×5 view), test on 50 **unseen** mazes; 3 seeds |

In [ ]:
from src.evaluation import load_experiment, compare_agents

METRICS_SHOWN = ["success", "path_efficiency", "steps", "mistakes", "hazards"]

def summary_table(name):
    exp = load_experiment(f"results/{name}")
    rows = []
    for r in exp["summary"]:
        row = {"agent": r["agent"], "split": r["split"], "seeds": r["n_seeds"]}
        for m in METRICS_SHOWN:
            row[m] = f"{r[m + '_mean']:.2f} ± {r[m + '_ci']:.2f}"
        rows.append(row)
    return pd.DataFrame(rows).set_index(["agent", "split"])

for name in ["fixed_maze", "hazard_maze", "terminal_hazards", "open_hazards", "generalization"]:
    print(f"\n=== {name}: {load_experiment(f'results/{name}')['experiment'].description}")
    display(summary_table(name))

### Are the differences real?

A permutation test asks how often a difference this large would appear if the agent labels were shuffled. With 5 seeds per agent the smallest possible p-value is 0.008; with 3 seeds it is 0.10, so the generalization experiment **cannot** reach conventional significance however large the gap.

In [ ]:
checks = [
    ("fixed_maze", "tabular_q", "random", "success", "train"),
    ("fixed_maze", "dqn", "tabular_q", "success", "train"),
    ("hazard_maze", "dqn", "tabular_q", "hazards", "train"),
    ("terminal_hazards", "dqn", "tabular_q", "success", "train"),
    ("generalization", "dqn", "tabular_q", "success", "unseen"),
    ("generalization", "tabular_q", "random", "success", "unseen"),
]
rows = []
for name, a, b, metric, split in checks:
    c = compare_agents(load_experiment(f"results/{name}")["results"], a, b, metric, split)
    rows.append({"experiment": name, "comparison": f"{a} − {b}", "metric": f"{metric} ({split})",
                 "difference": round(c["difference"], 2), "p-value": round(c["p_value"], 3)})
pd.DataFrame(rows)

### What the evidence supports

* **Both learning agents clearly beat random** on every fixed layout (p = 0.008): greedy success 1.00 for tabular Q and DQN on `fixed_maze`, `hazard_maze` and `open_hazards`, versus 0.06–0.40 for random.
* **Prioritized replay did not help** here: DQN+PER matched DQN on three experiments and failed one seed on `fixed_maze` (greedy policy looped).
* **Terminal hazards change what is rational.** DQN ended its greedy episode on a hazard in 4 of 5 seeds (success 0.20 vs tabular 1.00, p = 0.048) while its mistake count looked *lower* than random's. Mistake count alone is a misleading metric.
* **Hazard shortcuts:** tabular Q crossed a hazard in 4/5 seeds, DQN in 1/5, but **p = 0.21**: not established with 5 seeds.
* **Generalization:** on unseen mazes DQN succeeded 40–52% per seed versus 26–30% for tabular Q and 7% for random. DQN was ahead on every seed, but with 3 seeds the test cannot confirm it (p = 0.10). More seeds are needed before claiming it.

### Run a small experiment yourself

The full experiments take 20–60 minutes on the free CPU. This one finishes in under a minute:

In [ ]:
!python -m src.evaluation --experiment fixed_maze --seeds 2 --agents random tabular_q --root runs/quick

## 12. The lighter side: nicknames, achievements and the Hall of Shame

The commentator only **reads** the logs; it never changes a reward, an agent or a result. Verdicts follow the statistics, so a non-significant difference never gets celebrated.

In [ ]:
from src import fun

print("Meet the agents:")
for name in ["random", "tabular_q", "dqn", "dqn_per"]:
    print(f"  {fun.NICKNAMES[name]:<28} {fun.TAGLINES[name]}")

h = results["Tabular Q"]["history"]
mistakes = [{**m, "episode": e["episode"]} for e in h for m in e["mistake_log"]]
print("\n" + fun.format_hall_of_shame(fun.hall_of_shame(mistakes, top=5)))

print(f"\nAchievements for {fun.NICKNAMES['tabular_q']}:")
for a in fun.achievements(h):
    print(f"  episode {a['episode']:>4}: {a['title']}")

print("\nCommentary on a few episodes:")
for e in (h[0], h[len(h) // 2], h[-1]):
    print(f"  episode {e['episode']:>4} [{fun.mood(e)}]: {fun.episode_line(e)}")

print("\nThe verdict on the generalization experiment:")
c = compare_agents(load_experiment("results/generalization")["results"], "dqn", "tabular_q",
                   "success", "unseen")
print("  " + fun.verdict(c["p_value"], c["difference"], "dqn", "tabular_q"))

## 13. Charts of the saved results

`src/visualization.py` draws every chart in the dashboard. Bars show the final greedy success with 95% confidence intervals; the curves show greedy success during training (bands are 95% CIs across seeds, clipped to the possible range).

In [ ]:
from IPython.display import display
from src.visualization import experiment_bars_figure, experiment_curves_figure, reward_breakdown_figure

gen = load_experiment("results/generalization")
display(experiment_bars_figure(gen["summary"], "success"))
display(experiment_curves_figure(load_experiment("results/terminal_hazards")["curves"], "success"))
display(reward_breakdown_figure(results["DQN"]["history"]))

## 14. Launch the MIND dashboard (website)

Runs the Gradio app. In Colab, `share=True` prints a **public link** (valid for about 72 hours) that you can open in any browser or send to someone. The cell keeps running while the app is up; stop it with the ■ button.

This cell always serves the **latest code**: it closes any dashboard launched earlier in this session and reloads the project's code from disk first. **Use the new link it prints**; an older `gradio.live` link points to the older dashboard. The page header shows the version it runs, which should match the newest commit on GitHub.

Tabs: 📊 Results · 🏋️ Train · 🔥 Mistake history · 🎬 Watch it play · ⚔️ Q-learning vs DQN · ✅ Evaluate · 💾 Checkpoints · ℹ️ About

In [ ]:
import importlib
import gradio as gr
import src.envs, src.agents, src.replay, src.training, src.evaluation, src.fun, src.visualization, src.app

gr.close_all()   # stop dashboards launched earlier in this session (their links go dead)
# Reload in dependency order so files updated by `git pull` are used, not the copies
# Python loaded earlier in this session.
for module in (src.envs, src.agents, src.replay, src.training, src.evaluation,
               src.fun, src.visualization, src.app):
    importlib.reload(module)

print("Dashboard version:", src.app.code_version())
src.app.launch_app(share=True)